In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd

ROOT = Path.cwd().parents[1]
kbo = pd.read_csv(ROOT / "data/external/kbo_pitching.csv")
mlb = pd.read_csv(ROOT / "data/external/kbo_pitchers_mlb_lines.csv")

def rates(df):
    t = df.groupby("player_en")[["bf", "so", "bb", "ibb", "h", "hr"]].sum()
    return pd.DataFrame({
        "bf": t["bf"],
        "k_pct": t["so"] / t["bf"],
        "bb_pct": (t["bb"] - t["ibb"]) / t["bf"],
        "hr_pct": t["hr"] / t["bf"],
    })

k = rates(kbo).add_prefix("kbo_")
m = rates(mlb).add_prefix("mlb_")
both = k.join(m, how="inner")
print(f"{len(both)} pitchers in both leagues")
print(both[["kbo_bf", "mlb_bf"]].describe().round(0).to_string())

37 pitchers in both leagues
       kbo_bf  mlb_bf
count    37.0    37.0
mean   1202.0   737.0
std    1171.0   784.0
min      12.0     8.0
25%     473.0   175.0
50%     708.0   517.0
75%    1193.0  1013.0
max    4101.0  3398.0


In [2]:
MIN_BF = 200
q = both[(both["kbo_bf"] >= MIN_BF) & (both["mlb_bf"] >= MIN_BF)]
print(f"{len(q)} pitchers with {MIN_BF}+ BF in both leagues")
print()

for metric in ["k_pct", "bb_pct", "hr_pct"]:
    mv, kv = q[f"mlb_{metric}"], q[f"kbo_{metric}"]
    w_m = np.average(mv, weights=q["mlb_bf"])
    w_k = np.average(kv, weights=q["kbo_bf"])
    print(f"{metric:8s}  MLB {w_m:.3f} -> KBO {w_k:.3f}   "
          f"ratio {w_k / w_m:.2f}   corr {mv.corr(kv):.2f}")

24 pitchers with 200+ BF in both leagues

k_pct     MLB 0.185 -> KBO 0.198   ratio 1.07   corr 0.08
bb_pct    MLB 0.078 -> KBO 0.068   ratio 0.87   corr 0.64
hr_pct    MLB 0.034 -> KBO 0.019   ratio 0.57   corr 0.23


In [3]:
from scipy import stats
for metric in ["k_pct", "bb_pct", "hr_pct"]:
    r, p = stats.pearsonr(q[f"mlb_{metric}"], q[f"kbo_{metric}"])
    print(f"{metric:8s}  r = {r:+.3f}   p = {p:.3f}   n = {len(q)}")
print()
# Does a higher BF floor change the picture?
for floor in [200, 400, 600]:
    s = both[(both["kbo_bf"] >= floor) & (both["mlb_bf"] >= floor)]
    if len(s) < 8:
        continue
    print(f"BF >= {floor} (n={len(s)}): "
          f"K% r={s['mlb_k_pct'].corr(s['kbo_k_pct']):+.2f}  "
          f"BB% r={s['mlb_bb_pct'].corr(s['kbo_bb_pct']):+.2f}")

k_pct     r = +0.081   p = 0.707   n = 24
bb_pct    r = +0.642   p = 0.001   n = 24
hr_pct    r = +0.227   p = 0.285   n = 24

BF >= 200 (n=24): K% r=+0.08  BB% r=+0.64
BF >= 400 (n=14): K% r=-0.05  BB% r=+0.02
